# 33 · Load: Indeed Hiring Lab job postings index → `staging`

**Source card:** `indeed_hiring_lab.md`. Decision: D10: the national check beside Revelio's New York postings.

| Table | One row = | Expected rows |
|---|---|---|
| `stg_indeed__postings_national` | day × measure (total / new postings), U.S. | 4,872 |
| `stg_indeed__postings_category` | day × measure × Indeed job category (47), U.S. | 228,984 |
| `stg_indeed__postings_area` | day × area (New York State, New York metro) | 4,872 |

Indeed Hiring Lab `job_postings_tracker` (CC BY 4.0), snapshot 2026-10-12. Each value is an index, Feb 1, 2020 = 100, of a 7-day trailing average of postings on Indeed, seasonally adjusted series by series. Only the national file also has a not-adjusted version. Hiring Lab revised the whole history when it changed method (Nov 2024), so the files are snapshots, not a fixed history. Daily data from Feb 1, 2020 to Oct 2, 2026. The other 49 states and 380 or so metros are left out (D10 needs only the U.S. and New York).

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 60, "display.width", 220)
NB = "33-load-indeed-postings.ipynb"
conn = connect()

F = raw_files("indeed_hiring_lab")
def rd(stem: str):
    row = F[F.file.str.contains(f"/{stem}_")].iloc[-1]
    return pd.read_csv(row.path, dtype=str, keep_default_na=False), row.file

agg, f_agg = rd("aggregate_job_postings_US")
cat, f_cat = rd("job_postings_by_sector_US")
st, f_st = rd("state_job_postings_us")
met, f_met = rd("metro_job_postings_us")
for n, d in [("national", agg), ("category", cat), ("state", st), ("metro", met)]:
    print(f"{n:9} {d.shape} {d.date.min()} -> {d.date.max()}")

national  (4872, 5) 2020-02-01 -> 2026-10-02
category  (228984, 5) 2020-02-01 -> 2026-10-02
state     (124236, 3) 2020-02-01 -> 2026-10-02
metro     (1444548, 4) 2020-02-01 -> 2026-10-02


## 1. Checks on every file: dates, numbers, base day

Every series must have one value a day with no gaps, numbers only, and read exactly 100 on Feb 1, 2020 (the base).

In [2]:
def num(s):
    return pd.to_numeric(s, errors="raise").astype("float64")

days = pd.date_range("2020-02-01", agg.date.max(), freq="D")
def daily_ok(d, by):
    g = d.groupby(by).date.agg(["min", "max", "count", "nunique"])
    assert (g["min"] == "2020-02-01").all() and (g["max"] == agg.date.max()).all(), g
    assert (g["count"] == len(days)).all() and (g["nunique"] == len(days)).all(), "gap or repeated day"
    return len(g)

agg["index_sa"], agg["index_nsa"] = num(agg.indeed_job_postings_index_SA), num(agg.indeed_job_postings_index_NSA)
print("national series:", daily_ok(agg, "variable"))
cat["index_sa"] = num(cat.indeed_job_postings_index)
print("category series:", daily_ok(cat, ["variable", "display_name"]))

ny = st[st.state == "ny"].assign(area_code="NY", area_name="New York State", source_file=f_st)
nyc = met[met.cbsa_code == "35620"].assign(area_code="35620", area_name=lambda d: d.metro, source_file=f_met)
assert len(ny) and len(nyc) and nyc.metro.nunique() == 1
area = pd.concat([ny, nyc], ignore_index=True)
area["index_sa"] = num(area.indeed_job_postings_index)
print("area series:", daily_ok(area, "area_code"), "|", nyc.metro.iloc[0])

base = pd.concat([agg.loc[agg.date == "2020-02-01", ["index_sa", "index_nsa"]].melt().value,
                  cat.loc[cat.date == "2020-02-01", "index_sa"], area.loc[area.date == "2020-02-01", "index_sa"]])
print(f"Values on the base day: {base.nunique()} distinct ({base.unique()[:3]})")
assert (base == 100).all()
for n, s in [("national SA", agg.index_sa), ("national NSA", agg.index_nsa), ("category", cat.index_sa), ("area", area.index_sa)]:
    assert (s > 0).all(), n

national series: 2
category series: 94
area series: 2 | New York-Newark-Jersey City, NY-NJ-PA


Values on the base day: 1 distinct ([100.])


## 2. A first look: where each series stands on the latest day

In [3]:
last = agg.date.max()
display(pd.concat([agg[agg.date == last].set_index("variable")[["index_sa"]].rename(index=lambda v: f"U.S. {v}"),
                   area[area.date == last].set_index("area_name")[["index_sa"]]]))
c = cat[(cat.date == last) & (cat.variable == "total postings")].set_index("display_name").index_sa.sort_values()
print(f"Categories, total postings on {last} (Feb 2020 = 100): lowest and highest")
display(pd.concat([c.head(5), c.tail(5)]).to_frame())

,index_sa
U.S. total postings,103.84
U.S. new postings,99.40
New York State,90.02
"New York-Newark-Jersey City, NY-NJ-PA",85.02


Categories, total postings on 2026-10-02 (Feb 2020 = 100): lowest and highest


,index_sa
display_name,
Data & Analytics,63.14
"IT Infrastructure, Operations & Support",69.33
Media & Communications,70.91
Marketing,75.43
IT Systems & Solutions,76.36
Personal Care & Home Health,154.42
Civil Engineering,163.10
Mechanical Engineering,167.93
Therapy,187.14


## 3. Stage

`variable` ("total postings" / "new postings") becomes `measure` (`total` / `new`). Dates become `date`. Category names are Indeed's own; mapping them to SOC groups for the D10 check happens in core.

In [4]:
measure = {"total postings": "total", "new postings": "new"}
s_nat = (agg.assign(date=pd.to_datetime(agg.date), measure=agg.variable.map(measure), source_file=f_agg)
         [["date", "measure", "index_sa", "index_nsa", "source_file"]].sort_values(["measure", "date"]).reset_index(drop=True))
display(stage(s_nat, "stg_indeed__postings_national", ["date", "measure"], expected_rows=4_872, source_key="indeed_hiring_lab",
              notebook=NB, conn=conn, comment="Indeed Hiring Lab job postings index, U.S., daily, 7-day trailing average, Feb 1 2020 = 100, SA and NSA. CC BY 4.0."))

s_cat = (cat.assign(date=pd.to_datetime(cat.date), measure=cat.variable.map(measure), category=cat.display_name, source_file=f_cat)
         [["date", "measure", "category", "index_sa", "source_file"]].sort_values(["category", "measure", "date"]).reset_index(drop=True))
display(stage(s_cat, "stg_indeed__postings_category", ["date", "measure", "category"], expected_rows=228_984, source_key="indeed_hiring_lab",
              notebook=NB, conn=conn, comment="Indeed Hiring Lab job postings index by Indeed job category (47), U.S., daily, Feb 1 2020 = 100, SA. CC BY 4.0."))

s_area = (area.assign(date=pd.to_datetime(area.date))[["date", "area_code", "area_name", "index_sa", "source_file"]]
          .sort_values(["area_code", "date"]).reset_index(drop=True))
display(stage(s_area, "stg_indeed__postings_area", ["date", "area_code"], expected_rows=4_872, source_key="indeed_hiring_lab",
              notebook=NB, conn=conn, comment="Indeed Hiring Lab job postings index, New York State and New York-Newark-Jersey City metro (CBSA 35620), daily, Feb 1 2020 = 100, SA. CC BY 4.0."))
if conn is not None:
    conn.close()

  stg_indeed__postings_national: 4,872 rows x 5 cols, key ['date', 'measure'] unique | local data/interim/staging/stg_indeed__postings_national.parquet | DB: 4872


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_indeed__postings_national,4872,5,date + measure,indeed_hiring_lab,33-load-indeed-postings.ipynb,4872,2026-10-14T22:08:59+00:00


  stg_indeed__postings_category: 228,984 rows x 5 cols, key ['date', 'measure', 'category'] unique | local data/interim/staging/stg_indeed__postings_category.parquet | DB: 228984


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_indeed__postings_category,228984,5,date + measure + category,indeed_hiring_lab,33-load-indeed-postings.ipynb,228984,2026-10-14T22:09:00+00:00


  stg_indeed__postings_area: 4,872 rows x 5 cols, key ['date', 'area_code'] unique | local data/interim/staging/stg_indeed__postings_area.parquet | DB: 4872


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_indeed__postings_area,4872,5,date + area_code,indeed_hiring_lab,33-load-indeed-postings.ipynb,4872,2026-10-14T22:09:00+00:00
